In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.benchmark.audit import paired_gain
from src.benchmark.build_v1 import build_benchmark_v1
from src.benchmark.evaluate import DEFAULT_FEATURES, StyleSpace, evaluate, summarize, learning_curve
from src.ranking.lgbm_ranker import evaluate_ranker, train_ranker
from src.ranking.train_data import build_training_lineups


In [ ]:
df = pd.read_csv(ROOT / "data" / "processed" / "enron_messages.csv")
df["message_id"] = df["message_id"].astype(str)
features = [f for f in DEFAULT_FEATURES if (df[f] != 0).mean() >= 0.01]
space = StyleSpace(df, features)
print("features used:", features)
print("users", df["user_id"].nunique(), "rows", len(df))
print(df["user_split"].value_counts().to_dict(), df["time_split"].value_counts().to_dict())


In [ ]:
relabeled, bench_train = build_training_lineups(df, n_neg=9, history_frac=0.7, seed=0)
bench_v1 = build_benchmark_v1(df, n_neg=9, eval_splits=("val", "test"), max_items_per_user=50, seed=0)
print("train relabeled users", relabeled["user_id"].nunique())
print("train_lineups", len(bench_train), bench_train["tier"].value_counts().to_dict())
print("v1_lineups", len(bench_v1), bench_v1["tier"].value_counts().to_dict())


In [ ]:
out_dir = ROOT / "docs" / "findings"
out_dir.mkdir(parents=True, exist_ok=True)
model = train_ranker(space, bench_train, m=10, k=10.0, seed=0, n_estimators=60)
res_ranker = evaluate_ranker(space, bench_v1, model, m=10, k=10.0, seed=0)
summary_ranker = summarize(res_ranker, n_boot=200, seed=0)
res_ranker.to_csv(out_dir / "05_ranker_eval.csv", index=False)
summary_ranker.to_csv(out_dir / "05_ranker_summary.csv", index=False)
print("ranker summary:")
print(summary_ranker.round(3).to_string(index=False))


In [ ]:
base = evaluate(space, bench_v1, scorers=("population", "user_style"), m=10, k=10.0, seed=0)
base_summary = summarize(base, n_boot=200, seed=0)
base.to_csv(out_dir / "05_baseline_eval.csv", index=False)
base_summary.to_csv(out_dir / "05_baseline_summary.csv", index=False)
print("baseline comparison:")
print(base_summary.round(3).to_string(index=False))


In [ ]:
gain_ranker_vs_user = paired_gain(pd.concat([res_ranker.assign(scorer="ranker"), base], ignore_index=True), a="ranker", b="user_style", n_boot=200, seed=0)
learning_curve_df = learning_curve(space, bench_v1)
gain_ranker_vs_user.to_csv(out_dir / "05_paired_gain_ranker_vs_user.csv", index=False)
learning_curve_df.to_csv(out_dir / "05_learning_curve_ranker.csv")
print("paired_gain ranker minus user_style:")
print(gain_ranker_vs_user.round(3).to_string(index=False))
print("learning_curve:")
print(learning_curve_df.round(3).to_string())


In [ ]:
shuf_model = train_ranker(space, bench_train, m=10, k=10.0, seed=0, n_estimators=60, shuffle_labels=True)
shuf_res = evaluate_ranker(space, bench_v1, shuf_model, m=10, k=10.0, seed=0)
shuf_summary = summarize(shuf_res, n_boot=200, seed=0)
shuf_res.to_csv(out_dir / "05_shuffled_control_eval.csv", index=False)
shuf_summary.to_csv(out_dir / "05_shuffled_control_summary.csv", index=False)
print("shuffled-label control:")
print(shuf_summary.round(3).to_string(index=False))


In [ ]:
pop_model = train_ranker(space, bench_train, m=10, k=10.0, seed=0, n_estimators=60, drop_groups=("raw_z","deltas","abs_deltas","std_devs","llr_parts"))
pop_res = evaluate_ranker(space, bench_v1, pop_model, m=10, k=10.0, seed=0, drop_groups=("raw_z","deltas","abs_deltas","std_devs","llr_parts"))
pop_summary = summarize(pop_res, n_boot=200, seed=0)
pop_res.to_csv(out_dir / "05_population_only_control_eval.csv", index=False)
pop_summary.to_csv(out_dir / "05_population_only_control_summary.csv", index=False)
print("population-only control:")
print(pop_summary.round(3).to_string(index=False))
